# M03 · PID with anti-windup, filters and step-response metrics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/04_control_motion/M03_pid_with_anti_windup_filters_and_step_response_m.ipynb)

**Track:** Control and motion · **Skill axis (sheet):** Control and motion · **Time:** about 65 min

**Skill Hub concepts:** PID and anti-windup; Discretizing a first-order filter; Step response metrics

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("M03")

## Map: a production PID, term by term

```
 setpoint ─(+)─ e ─┬─ kp·e ─────────────────────────────┐
           (-)     ├─ ki·Σe·dt   (frozen while saturated │──(+)── clamp[u_min, u_max] ── u ──> motor
            ↑      └ (on measurement) -kd·LPF(dy/dt) ────┘                                     │
            └──────────────────────────────── measurement y <──────────────────────────────────┘
```

| Problem | Fix |
|---|---|
| Derivative kick on setpoint steps | differentiate the **measurement**, not the error |
| Noise amplified by the derivative | first-order low-pass on the derivative |
| **Windup**: integrator keeps growing while the output is saturated, then overshoots | conditional integration (clamping) or back-calculation |

Exact (matched) discretization of a first-order low-pass with cutoff $f_c$, sampled every $\Delta t$:
$$y_k = y_{k-1} + \alpha\,(x_k - y_{k-1}), \qquad \alpha = 1 - e^{-2\pi f_c \Delta t}$$

Plant used here (motor speed): $\tau \dot\omega = -\omega + K u$, $\tau = 0.5$ s, $K = 10$ rad/s per volt, $|u| \le 12$ V,
simulated exactly: $\omega_{k+1} = a\,\omega_k + (1-a) K u_k$, $a = e^{-\Delta t/\tau}$.

| Metric | Definition used here |
|---|---|
| Rise time | time from first reaching 10% of final to first reaching 90% |
| Overshoot | $(\max y - y_\infty)/y_\infty \times 100\%$, 0 if never above |
| Settling time (2%) | first time after which $y$ stays within 2% of $y_\infty$ |
| First-order reference | rise $= \tau \ln 9$, settling $= \tau \ln 50$ |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

## Task 1: the filter coefficient

In [ ]:
def lowpass_alpha(fc, dt):
    raise NotImplementedError  # TODO

lab.check("alpha", lowpass_alpha)

In [ ]:
%%writefile m03_alpha.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/m03.h
// Grade it with lab.check("alpha", "m03_alpha.c"). For C++, name the file .cpp in both places.
#include "m03.h"

double lowpass_alpha(double fc, double dt) { (void)fc; (void)dt; return 0; /* TODO */ }

In [ ]:
# lab.check("alpha", "m03_alpha.c")  # uncomment after writing the C/C++ cell above

## Task 2: the PID
Derivative on measurement with the low-pass above (no kick on the first call), output clamp, and conditional
integration: skip the integrator update when it would push a saturated output further into saturation.

In [ ]:
class PID:
    def __init__(self, kp, ki, kd, dt, u_min, u_max, d_cutoff_hz=50.0):
        self.kp, self.ki, self.kd, self.dt = kp, ki, kd, dt
        self.u_min, self.u_max = u_min, u_max
        self.alpha = lowpass_alpha(d_cutoff_hz, dt)
        self.integ, self.d_f, self.prev_y = 0.0, 0.0, None

    def update(self, setpoint, y):
        raise NotImplementedError  # TODO

lab.check("pid", PID)

In [ ]:
%%writefile m03_pid.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/m03.h
// Grade it with lab.check("pid", "m03_pid.c", "m03_alpha.c"). For C++, name the file .cpp in both places.
#include "m03.h"

void pid_init(Pid *p, double kp, double ki, double kd, double dt, double u_min, double u_max, double d_cutoff_hz) {
    p->kp = kp; p->ki = ki; p->kd = kd; p->dt = dt; p->u_min = u_min; p->u_max = u_max;
    p->alpha = lowpass_alpha(d_cutoff_hz, dt);
    p->integ = 0; p->d_f = 0; p->prev_y = 0; p->has_prev = false;
}

double pid_update(Pid *p, double setpoint, double y) {
    (void)p; (void)setpoint; (void)y;
    return 0; /* TODO: filtered derivative on measurement, conditional integration, clamp */
}

In [ ]:
# lab.check("pid", "m03_pid.c", "m03_alpha.c")  # uncomment after writing the C/C++ cell above

## Task 3: step-response metrics

In [ ]:
def step_metrics(t, y, final):
    """Return (rise_time, overshoot_percent, settling_time) with the definitions in the table."""
    raise NotImplementedError  # TODO

lab.check("metrics", step_metrics)

In [ ]:
%%writefile m03_metrics.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/m03.h
// Grade it with lab.check("metrics", "m03_metrics.c"). For C++, name the file .cpp in both places.
#include "m03.h"

void step_metrics(const double *t, const double *y, size_t n, double final_value, double *rise_time, double *overshoot_percent, double *settling_time) {
    (void)t; (void)y; (void)n; (void)final_value;
    *rise_time = 0; *overshoot_percent = 0; *settling_time = 0; /* TODO: 10-90 % rise, overshoot, 2 % settling */
}

In [ ]:
# lab.check("metrics", "m03_metrics.c")  # uncomment after writing the C/C++ cell above

Try it: a 100 rad/s step, then an unreachable 150 rad/s, then back to 50. Compare with the integrator always on.

In [ ]:
def simulate(pid_factory, profile, dt=1e-3, tau=0.5, K=10.0):
    a, w, ws, us = math.exp(-dt / tau), 0.0, [], []
    pid = pid_factory()
    for sp in profile:
        u = pid.update(sp, w)
        w = a * w + (1 - a) * K * u
        ws.append(w); us.append(u)
    return np.array(ws), np.array(us)

profile = [100.0] * 2000 + [150.0] * 4000 + [50.0] * 3000
try:
    ws, us = simulate(lambda: PID(0.2, 2.0, 0.0, 1e-3, -12, 12), profile)
    t = np.arange(len(profile)) * 1e-3
    fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    ax[0].plot(t, profile, "k--", label="setpoint"); ax[0].plot(t, ws, label="speed"); ax[0].legend(); ax[0].set_ylabel("rad/s")
    ax[1].plot(t, us); ax[1].set_ylabel("volts"); ax[1].set_xlabel("s")
    plt.show()
    print("metrics of the first step:", step_metrics(t[:2000], ws[:2000], 100.0))
except NotImplementedError:
    print("finish Tasks 1 to 3 first")

## Checkpoint: Math
**M1 (Apply).** $\alpha$ for a 10 Hz derivative filter at 1 kHz (4 decimals)?

**M2 (Apply).** 10 to 90% rise time of the open-loop plant ($\tau = 0.5$ s) in seconds?

**M3 (Analyze).** P-only control ($k_p = 0.5$) of the plant ($K = 10$) with a 100 rad/s setpoint, no saturation.
Steady-state error in rad/s?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why put the derivative on the measurement?
- a) it reduces noise
- b) a setpoint step makes d(error)/dt infinite for one sample (derivative kick); the measurement cannot jump
- c) it removes steady-state error

**K2 (Analyze).** Explain integrator windup with the 150 rad/s segment above (2 s to 6 s), and compare conditional integration
with back-calculation.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `m03.h`).
3. In `roboprep/checks/m03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()